# AkasicDB 하이브리드 검색: 그래프 + 전문 검색 + 벡터

이 노트북에서는 합성 상품 데이터를 AkasicDB에 적재하고, **그래프(Cypher)**, **전문 검색(BM25, texton)**, **벡터 유사도(vectoron)**, **SQL 필터**를 하나의 SQL 쿼리로 함께 실행합니다. 마지막으로 PostgreSQL의 `EXPLAIN`으로 실행 계획을 확인합니다.

상품 설명은 texton의 한국어 형태소 분석기(lindera + ko-dic)로 색인합니다. "러닝화를", "방수가"처럼 조사가 붙은 표현도 "러닝화", "방수"로 검색되는 것을 Step 9에서 확인합니다.

## 준비물

* AkasicDB 인스턴스
    * [db.akasic.cloud](https://db.akasic.cloud)에서 Playground 인스턴스를 만들거나,
    * [설치 문서](https://docs.graphai.io/akasicdb-docs/latest/installation/installation/)를 따라 로컬에 설치합니다.
* 연결 문자열
    * Google Colab: 왼쪽 사이드바의 열쇠 아이콘을 눌러 `DB_DSN`이라는 이름의 Secret으로 저장합니다.
    * 그 밖의 환경: 환경 변수 `DATABASE_URL`로 지정합니다.
* (선택) GPU 런타임. Colab에서는 **런타임 → 런타임 유형 변경 → T4 GPU**를 고릅니다. GPU가 없으면 데이터 규모를 줄여 CPU로 실행합니다.

## Step 1: 패키지 설치

In [ ]:
%pip install --quiet akasicdb "psycopg[binary]" "sentence-transformers>=2.7" "transformers>=4.51" accelerate

## Step 2: DB 연결과 확장 활성화

환경 변수 `DATABASE_URL`이 있으면 그 값을, 없으면 Colab Secrets의 `DB_DSN`을 사용합니다.

`akasicdb` 확장은 데이터베이스마다 한 번 활성화해야 합니다. `CASCADE`는 의존 확장(`vectoron`)을 함께 설치합니다. 이미 활성화된 데이터베이스에서 다시 실행해도 안전합니다.

In [ ]:
import os

import psycopg
from akasicdb.psycopg import register_vector


def get_dsn() -> str:
    dsn = os.getenv("DATABASE_URL")
    if dsn:
        return dsn
    try:
        from google.colab import userdata
    except ImportError:
        raise RuntimeError("환경 변수 DATABASE_URL을 설정하거나 Colab Secrets에 DB_DSN을 저장해 주세요.")
    return userdata.get("DB_DSN")


conn = psycopg.connect(get_dsn(), autocommit=True)
conn.execute("CREATE EXTENSION IF NOT EXISTS akasicdb CASCADE")
conn.execute("SELECT akasicdb_admin.initialize()")
register_vector(conn)

print(conn.execute("SELECT version()").fetchone()[0])

## Step 3: 스키마 생성

노트북을 다시 실행할 수 있도록 이전에 만든 그래프와 테이블을 먼저 지웁니다. 상품 설명 임베딩은 Qwen3-Embedding-0.6B의 출력 차원에 맞춰 `vectoron.vector(1024)`로 저장합니다.

In [ ]:
for stmt in (
    "SELECT akasicdb.delete_graph('retail_graph', true)",
    "SELECT akasicdb.undefine_graph('retail_graph')",
):
    try:
        conn.execute(stmt)
    except psycopg.Error:
        pass  # 그래프가 없으면 건너뜁니다.

conn.execute("DROP TABLE IF EXISTS products_std, products, brands CASCADE")

conn.execute("""
CREATE TABLE brands (
  b_id  integer PRIMARY KEY,
  name  text    NOT NULL,
  hot   boolean NOT NULL DEFAULT false
)""")

conn.execute("""
CREATE TABLE products (
  id       integer PRIMARY KEY,
  b_id     integer NOT NULL REFERENCES brands(b_id),
  title    text    NOT NULL,
  category text    NOT NULL,
  price    integer NOT NULL,
  descr    text    NOT NULL,
  emb      vectoron.vector(1024)
)""")
print("ok")

## Step 4: 합성 데이터 생성

GPU가 있으면 상품 10,000개, 없으면 2,000개를 만듭니다. 브랜드 50개 중 10개는 인기(`hot`) 브랜드이고, 인기 브랜드 상품은 모든 카테고리에 고르게 들어갑니다.

상품 설명은 계절, 품목, 특징을 조합한 문장입니다. "러닝화를", "트레킹화는", "방수가"처럼 조사가 붙은 형태가 섞여 있습니다.

In [ ]:
import random

import torch

if torch.cuda.is_available():
    DEVICE, DTYPE, N_PRODUCTS = "cuda", torch.float16, 10_000
else:
    DEVICE, DTYPE, N_PRODUCTS = "cpu", torch.float32, 2_000
print(f"장치: {DEVICE}, 상품 수: {N_PRODUCTS:,}")

random.seed(42)
N_BRANDS = 50

seasons = ["여름", "겨울", "사계절", "봄가을"]
items = ["러닝화", "운동화", "샌들", "트레킹화", "슬립온"]
feats = ["통기성이 좋은", "가볍고 시원한", "쿠션이 좋은", "방수가 되는"]
cats = ["shoes", "apparel", "bag", "acc", "outdoor"]


def josa(word: str, with_batchim: str, without_batchim: str) -> str:
    """받침 유무에 맞는 조사를 붙입니다. 예: 러닝화를, 샌들을"""
    has_batchim = (ord(word[-1]) - 0xAC00) % 28 != 0
    return word + (with_batchim if has_batchim else without_batchim)


templates = [
    lambda s, i, f: f"{s} 시즌 {josa(i, '을', '를')} 찾는 분께 추천하는 {f} 제품입니다.",
    lambda s, i, f: f"{josa(i, '은', '는')} {f} 소재라 {s}에 신기 좋습니다.",
    lambda s, i, f: f"{f} {josa(i, '이', '가')} {s} 내내 편안합니다.",
]

brands = [(b, f"brand_{b}", b % 5 == 0) for b in range(1, N_BRANDS + 1)]

products = []
for g in range(1, N_PRODUCTS + 1):
    season = seasons[(g // 7) % len(seasons)]
    item = items[(g // 3) % len(items)]
    feat = feats[g % len(feats)]
    descr = templates[(g // 11) % len(templates)](season, item, feat)
    products.append((
        g,                                   # id
        (g // 5) % N_BRANDS + 1,             # b_id: 카테고리와 독립적으로 배정
        f"product_{g}",                      # title
        cats[g % len(cats)],                 # category
        10_000 + random.randrange(190_000),  # price
        descr,
    ))

hot_brands = {b for b, _, hot in brands if hot}
hot_shoes = sum(1 for p in products if p[1] in hot_brands and p[3] == "shoes")
print(f"인기 브랜드의 shoes 상품: {hot_shoes}개")
for p in products[:3]:
    print(p)

## Step 5: 임베딩 생성

Qwen3-Embedding-0.6B는 다국어 임베딩 모델입니다. 상품 설명은 instruction 없이 인코딩하고, 검색 질의는 `prompt_name="query"`로 인코딩합니다. `normalize_embeddings=True`로 정규화하면 L2 거리(`<->`) 순서가 코사인 유사도 순서와 같아집니다.

T4 GPU에서 10,000건은 수 분, CPU에서 2,000건은 수 분에서 십여 분 걸립니다.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device=DEVICE,
    model_kwargs={"torch_dtype": DTYPE},
    tokenizer_kwargs={"padding_side": "left"},
)

embs = model.encode(
    [p[5] for p in products],
    batch_size=64 if DEVICE == "cuda" else 16,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(embs.shape)

## Step 6: 데이터 적재

In [ ]:
with conn.cursor() as cur:
    cur.executemany("INSERT INTO brands VALUES (%s, %s, %s)", brands)

    rows = [(*p, emb) for p, emb in zip(products, embs)]
    BATCH = 1000
    for i in range(0, len(rows), BATCH):
        cur.executemany(
            "INSERT INTO products (id, b_id, title, category, price, descr, emb) "
            "VALUES (%s, %s, %s, %s, %s, %s, %s)",
            rows[i:i + BATCH],
        )
        print(f"{min(i + BATCH, len(rows)):,} / {len(rows):,}")

conn.execute("ANALYZE brands, products")
print("done")

## Step 7: 인덱스 생성

세 종류의 인덱스를 만듭니다.

* **vectoron**: 임베딩 벡터의 근사 최근접 검색(ANN)
* **texton**: 상품 설명의 전문 검색(BM25). 행 전체(`products.*`)를 대상으로 하는 whole-row 인덱스이며, `mapping`에서 `descr` 필드를 한국어 형태소 분석기(lindera + ko-dic)로 색인하도록 지정합니다. `@@` 검색 연산자는 이 인덱스를 사용합니다.
* **B-tree**: 카테고리 필터

In [ ]:
conn.execute("CREATE INDEX products_emb_idx ON products USING vectoron (emb vector_l2_ops)")

conn.execute("""
CREATE INDEX products_texton_idx
  ON products
  USING texton ((products.*))
  WITH (mapping='{"fields":{
    "descr":{"type":"text","analyzer":{"tokenizer":{"lindera":{"dictionary":"ko-dic"}}}}
  }}')""")

conn.execute("CREATE INDEX products_cat_idx ON products (category)")
print("ok")

## Step 8: 그래프 정의

기존 `products`, `brands` 테이블에서 그래프의 정점과 간선을 정의합니다. 데이터를 다시 적재하지 않습니다.

* 정점: `v_product`(상품), `v_brand`(브랜드)
* 간선: `e_made_by`(상품 → 브랜드)

In [ ]:
conn.execute("SELECT akasicdb.define_graph('retail_graph')")

conn.execute("""
SELECT akasicdb.define_vertex('retail_graph', 'v_product',
  ARRAY['product_id integer', 'title text'],
  'products',
  'SELECT id AS product_id, title FROM products')""")

conn.execute("""
SELECT akasicdb.define_vertex('retail_graph', 'v_brand',
  ARRAY['b_id integer', 'name text', 'hot boolean'],
  'brands',
  'SELECT b_id, name, hot FROM brands')""")

conn.execute("""
SELECT akasicdb.define_edge('retail_graph', 'e_made_by',
  'v_product', 'v_brand',
  null,
  'SELECT null FROM products p, brands b WHERE p.b_id = b.b_id',
  'products p', 'brands b')""")

conn.execute("SELECT akasicdb.create_graph('retail_graph')")
print("graph ready")

## Step 9: 한국어 형태소 분석 효과 확인

비교를 위해 같은 상품 설명을 언어 중립 `standard` analyzer로 색인한 테이블 `products_std`를 만들고, 같은 검색어의 결과 건수를 비교합니다.

`standard`는 공백과 문장부호로만 단어를 나누므로 "러닝화를"을 한 단어로 색인하고, "러닝화"로는 찾지 못합니다. ko-dic은 형태소 단위로 나누므로 조사가 붙은 표현도 찾습니다.

In [ ]:
conn.execute("CREATE TABLE products_std AS SELECT id, descr FROM products")
conn.execute("""
CREATE INDEX products_std_texton_idx
  ON products_std
  USING texton ((products_std.*))
  WITH (mapping='{"fields":{"descr":{"type":"text"}}}')""")

print(f"{'검색어':<8}{'ko-dic':>8}{'standard':>10}")
for word in ["러닝화", "방수", "트레킹화"]:
    ko = conn.execute(
        "SELECT count(*) FROM products p WHERE p @@ akasicdb.text_query(%s)", (word,)
    ).fetchone()[0]
    std = conn.execute(
        "SELECT count(*) FROM products_std s WHERE s @@ akasicdb.text_query(%s)", (word,)
    ).fetchone()[0]
    print(f"{word:<8}{ko:>8}{std:>10}")

## Step 10: 하이브리드 질의

하나의 SQL 쿼리에서 네 가지 조건을 함께 사용합니다.

1. **그래프(Cypher)**: 인기 브랜드(`hot = true`)가 만든 상품
2. **SQL 필터**: `category = 'shoes'`
3. **전문 검색(BM25)**: 설명에 "여름"과 "러닝화"가 모두 들어간 상품. `akasicdb.text_score(p)`로 BM25 점수도 함께 봅니다.
4. **벡터 유사도**: "여름에 신을 시원한 러닝화"와 의미가 가까운 순서로 정렬

In [ ]:
q_emb = model.encode(
    ["여름에 신을 시원한 러닝화"],
    prompt_name="query",
    normalize_embeddings=True,
)[0]

HYBRID_SQL = """
SELECT p.id, p.title, p.price, akasicdb.text_score(p) AS bm25, left(p.descr, 40) AS descr
FROM   products p
JOIN   akasicdb.cypher('retail_graph', $$
           MATCH (pr:v_product)-[:e_made_by]->(b:v_brand)
           WHERE b.hot = true
           RETURN pr.product_id AS product_id
       $$) AS g (product_id integer)
       ON g.product_id = p.id                        -- 그래프 (Cypher)
WHERE  p.category = 'shoes'                          -- SQL 필터
  AND  p @@ akasicdb.text_query('여름 AND 러닝화')     -- 전문 검색 (BM25)
ORDER  BY p.emb <-> %s::vectoron.vector               -- 벡터 유사도 (L2)
LIMIT  10
"""

rows = conn.execute(HYBRID_SQL, (q_emb,)).fetchall()
for row in rows:
    print(row)
print(f"하이브리드 결과: {len(rows)}건")

## Step 11: 실행 계획 확인

표준 PostgreSQL `EXPLAIN`으로 하이브리드 질의의 실행 계획을 봅니다.

In [ ]:
plan = conn.execute("EXPLAIN (ANALYZE, BUFFERS) " + HYBRID_SQL, (q_emb,)).fetchall()
for (line,) in plan:
    # 질의 벡터(1024차원)가 그대로 찍히는 긴 줄은 잘라서 보여줍니다.
    print(line if len(line) <= 160 else line[:160] + " ...")

### 계획에서 읽을 포인트

1. **전문 검색**: `Index Scan using products_texton_idx on products p`가 보이면 texton 인덱스로 BM25 검색을 수행한 것입니다. texton의 `@@` 조건은 항상 이 인덱스 스캔으로 평가되며, `category` 조건은 같은 노드의 `Filter`로 적용됩니다.
2. **벡터 정렬**: `Sort` 노드에서 `p.emb <-> ...` 거리로 정렬하면, 앞 단계의 조건을 통과한 후보만 거리를 계산한 것입니다. 전문 검색과 그래프 조건으로 후보가 크게 줄어든 경우 플래너는 이 경로를 고릅니다.
3. **그래프**: Cypher 질의는 그래프 토폴로지 테이블(`v_product_topo`, `v_brand_topo`, `e_made_by_topo`)에 대한 조인으로 펼쳐져, 다른 조건과 함께 하나의 계획으로 최적화됩니다.

조건(`WHERE b.hot`, `category`, 전문 검색어)을 바꿔 가며 실행 계획이 어떻게 달라지는지 비교해 보세요.